# CT-CLIP (CT_LiPro_v2) — predictions + Grad-CAM on the best cases

Runs the ReXGround `dataset_2.json` nodule/GGO cases through the CT-LiPro classifier,
picks the highest-confidence cases per pathology, and renders CT / ground-truth /
Grad-CAM triptychs for them.

Reuses the tested pipeline code in `models/ct-clip/scripts/` rather than
re-implementing preprocessing or Grad-CAM here — this notebook is a thin,
interactive driver around:

- `rexground_pilot_pipeline.py` — model loading, preprocessing, Grad-CAM
- `rexground_full_pipeline.py` — full-dataset case listing, prediction loop, case selection
- `visualize_gradcam_top20.py` — GT-mask alignment + best-Dice-slice picking

**Preprocessing note:** `preprocess_ct` clips HU to [-1000, 1000] *before* resampling,
matching `data_inference_nii.py::CTReportDatasetinfer` — the dataset class
`ct_lipro_train.py` actually used to train this checkpoint. (An earlier version that
clipped *after* resampling, matching the separate zero-shot-pretraining dataset class,
measurably hurt AUROC on this checkpoint — see the sanity-check cell near the bottom.)


In [ ]:
import sys
from pathlib import Path

SCRIPTS_DIR = Path("/home/chest_ct/code/models/ct-clip/scripts")
sys.path.insert(0, str(SCRIPTS_DIR))

import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
%matplotlib inline

from rexground_pilot_pipeline import (
    load_ctclip_classifier, preprocess_ct, run_gradcam,
    VOLUME_DIR, SEG_DIR, TRAIN_META, VALID_META,
    LUNG_NODULE_IDX, LUNG_OPACITY_IDX, DEVICE, ROOT,
)
from rexground_full_pipeline import build_full_case_list, run_predictions, select_top20, gradcam_overlap
from visualize_gradcam_top20 import aligned_gt_mask, best_dice_slice

print("device:", DEVICE)


## 1. Build the case list

Every `dataset_2.json` case tagged `2c` (GGO) and/or `2d` (nodule) that has a local
volume, segmentation mask, and DICOM metadata (RescaleSlope/Intercept/spacing) available.


In [ ]:
cases = build_full_case_list()
n_nodule = sum(1 for c in cases if c["kind"] == "nodule")
n_ggo = sum(1 for c in cases if c["kind"] == "ggo")
n_mixed = sum(1 for c in cases if c["kind"] == "mixed")
print(f"{len(cases)} total cases  ({n_nodule} nodule-only, {n_ggo} ggo-only, {n_mixed} mixed)")


## 2. Load the model


In [2]:
model, tokenizer = load_ctclip_classifier()
train_meta = pd.read_csv(TRAIN_META).set_index("VolumeName")
valid_meta = pd.read_csv(VALID_META).set_index("VolumeName")

NameError: name 'load_ctclip_classifier' is not defined

## 3. Run predictions

Set `N_CASES = None` to run the entire case list (~2,000 cases, ~1–2 s/case on GPU).
Set it to a small number (e.g. `50`) for a quick trial run first.


In [1]:
N_CASES = 30  # ~1.5-3s/case on GPU, so this finishes in under a minute; None = run every available case

cases_by_lesion_size = sorted(
    cases, key=lambda c: -sum(c["case"].get("pixels", {}).values())
)
case_subset = cases if N_CASES is None else cases_by_lesion_size[:N_CASES]

predictions_df = run_predictions(model, tokenizer, case_subset, train_meta, valid_meta)
predictions_df.head()


NameError: name 'cases' is not defined

In [ ]:
OUT_DIR = ROOT / "rexground_predictions/notebook_run"
OUT_DIR.mkdir(parents=True, exist_ok=True)
predictions_df.to_csv(OUT_DIR / "predictions.csv", index=False)
print("saved:", OUT_DIR / "predictions.csv")


## 4. Select the best cases

Top 10 nodule + top 10 GGO cases by raw target-pathology logit (not the sigmoid
probability — logits rank identically but aren't compressed near 0.5, see the
calibration note at the end of this notebook).


In [ ]:
N_BEST = 3  # how many cases to run Grad-CAM + visualize

valid_preds = predictions_df[predictions_df["error"].isna()] if "error" in predictions_df.columns else predictions_df

nodule_pool = valid_preds[valid_preds["kind"].isin(["nodule", "mixed"])].copy()
nodule_pool["gradcam_target"] = "nodule"
nodule_pool["target_logit"] = nodule_pool["logit_lung_nodule"]

ggo_pool = valid_preds[valid_preds["kind"].isin(["ggo", "mixed"])].copy()
ggo_pool["gradcam_target"] = "ggo"
ggo_pool["target_logit"] = ggo_pool["logit_lung_opacity"]

combined = pd.concat([nodule_pool, ggo_pool]).sort_values("target_logit", ascending=False)
top_cases = combined.head(N_BEST).to_dict("records")

top_df = pd.DataFrame(top_cases)
top_df[["name", "kind", "gradcam_target", "target_logit", "logit_lung_nodule", "logit_lung_opacity"]]


## 5. Grad-CAM on the best cases


In [ ]:
import nibabel as nib

gradcam_results = []

for rec in top_cases:
    name = rec["name"]
    target = rec["gradcam_target"]
    target_idx = LUNG_NODULE_IDX if target == "nodule" else LUNG_OPACITY_IDX
    meta_row = train_meta.loc[name] if name in train_meta.index else valid_meta.loc[name]

    image_tensor, crop_info = preprocess_ct(VOLUME_DIR / name, meta_row)
    cam, probs = run_gradcam(model, tokenizer, image_tensor, target_idx)

    seg_nii = nib.load(str(SEG_DIR / name))
    seg = seg_nii.get_fdata()
    if seg.ndim == 4:
        seg = seg[0]
    seg_mask = seg > 0

    best_dice, best_pct, pointing_hit = gradcam_overlap(cam, seg_mask, crop_info, meta_row)
    target_prob = float(probs[target_idx])

    print(f"{target:6s} {name:28s} prob={target_prob:.3f} "
          f"best_dice@{best_pct}%={best_dice:.4f} pointing_hit={pointing_hit}")

    gradcam_results.append({
        **rec, "target_prob": target_prob, "cam": cam, "crop_info": crop_info,
        "meta_row": meta_row, "gradcam_best_dice": best_dice,
        "gradcam_best_pct": best_pct, "gradcam_pointing_hit": pointing_hit,
    })


## 6. Visualize

CT / ground-truth / Grad-CAM triptych for each case, at the slice with the highest
per-slice Dice at that case's own best CAM threshold (not just the slice with the
most ground-truth voxels).


In [ ]:
import nibabel as nib
from scipy.ndimage import zoom

TARGET_SHAPE_XYZ = (480, 480, 240)


def cam_to_full_resolution(cam_zxy, crop_info, full_shape):
    """Undo the crop+pad step, then resample up to the raw CT's native
    grid -- the CT-CLIP equivalent of Merlin's simple `zoom(cam, ct.shape
    / cam.shape)`. Merlin's preprocessing only resizes (no crop/pad), so
    a single zoom suffices there; CT-CLIP's preprocessing crops/pads
    around a resampled canvas, so that step has to be inverted first or
    the CAM ends up spatially offset from the true anatomy.
    """
    cam_xyz = np.transpose(cam_zxy, (1, 2, 0))  # (X, Y, Z), matches TARGET_SHAPE_XYZ

    h, w, d = crop_info["orig_shape"]           # resampled, pre-crop shape
    dh, dw, dd = TARGET_SHAPE_XYZ
    hs, ws, ds = crop_info["h_start"], crop_info["w_start"], crop_info["d_start"]
    ph, pw, pd = crop_info["pad_h_before"], crop_info["pad_w_before"], crop_info["pad_d_before"]
    ch, cw, cd = min(dh, h), min(dw, w), min(dd, d)  # actual cropped/copied extent

    resampled_canvas = np.zeros((h, w, d), dtype=cam_xyz.dtype)
    resampled_canvas[hs:hs + ch, ws:ws + cw, ds:ds + cd] = (
        cam_xyz[ph:ph + ch, pw:pw + cw, pd:pd + cd]
    )

    zoom_factors = (full_shape[0] / h, full_shape[1] / w, full_shape[2] / d)
    return zoom(resampled_canvas, zoom_factors, order=1)


def show_case(result):
    name = result["name"]
    target = result["gradcam_target"]
    cam_zxy = result["cam"]
    crop_info = result["crop_info"]
    meta_row = result["meta_row"]

    ct_path = VOLUME_DIR / name
    gt_path = SEG_DIR / name

    ct_nii = nib.load(str(ct_path))
    gt_nii = nib.load(str(gt_path))
    ct = ct_nii.get_fdata()
    gt = gt_nii.get_fdata()
    if gt.ndim == 4:
        gt = gt[0]
    gt_mask = gt > 0

    # --- low-res threshold sweep, in the model's own preprocessed space,
    #     exactly mirroring the Merlin script's structure ---
    gt_small = aligned_gt_mask(name, crop_info, meta_row)  # already aligned to cam's grid
    cam_xyz_small = np.transpose(cam_zxy, (1, 2, 0))

    best_dice, best_threshold = 0.0, 0.0
    for p in [70, 75, 80, 85, 90, 95]:
        threshold = np.percentile(cam_xyz_small, p)
        cam_mask = cam_xyz_small >= threshold
        inter = np.logical_and(cam_mask, gt_small).sum()
        dice = 2 * inter / (cam_mask.sum() + gt_small.sum() + 1e-8)
        if dice > best_dice:
            best_dice, best_threshold = dice, threshold

    max_point = np.unravel_index(np.argmax(cam_xyz_small), cam_xyz_small.shape)
    pointing_hit = bool(gt_small[max_point]) if gt_small.sum() > 0 else None

    # --- resample CAM up to the CT's native full resolution ---
    cam_full = cam_to_full_resolution(cam_zxy, crop_info, ct.shape)

    # --- full-resolution metrics ---
    cam_mask_full = cam_full >= best_threshold
    intersection = np.logical_and(cam_mask_full, gt_mask).sum()
    union = np.logical_or(cam_mask_full, gt_mask).sum()
    dice_full = 2 * intersection / (cam_mask_full.sum() + gt_mask.sum() + 1e-8)
    iou_full = intersection / (union + 1e-8)
    tp = intersection
    fp = np.logical_and(cam_mask_full, ~gt_mask).sum()
    fn = np.logical_and(~cam_mask_full, gt_mask).sum()
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)

    print(f"{name}  |  target={target}  |  low-res best Dice={best_dice:.4f} (thr@{best_threshold:.3f})  "
          f"|  pointing_hit={pointing_hit}")
    print(f"  full-res  Dice={dice_full:.4f}  IoU={iou_full:.4f}  "
          f"Precision={precision:.4f}  Recall={recall:.4f}")

    # --- visualize at full resolution, same layout as the Merlin notebook ---
    slice_idx = np.argmax(gt_mask.sum(axis=(0, 1)))

    plt.figure(figsize=(18, 6))

    plt.subplot(1, 3, 1)
    plt.imshow(ct[:, :, slice_idx], cmap="gray")
    plt.title(f"{name}\nCT slice {slice_idx}")
    plt.axis("off")

    plt.subplot(1, 3, 2)
    plt.imshow(ct[:, :, slice_idx], cmap="gray")
    plt.imshow(np.ma.masked_where(gt_mask[:, :, slice_idx] == 0, gt_mask[:, :, slice_idx]),
               cmap="Reds", alpha=0.6)
    plt.title("Ground truth")
    plt.axis("off")

    plt.subplot(1, 3, 3)
    plt.imshow(ct[:, :, slice_idx], cmap="gray")
    plt.imshow(cam_full[:, :, slice_idx], cmap="jet", alpha=0.5)
    plt.title(f"CT-CLIP Grad-CAM ({target})\nfull-res Dice={dice_full:.4f}")
    plt.axis("off")

    plt.tight_layout()
    plt.show()


### One-by-one viewer

Browse the selected cases individually with **Prev / Next**, or jump straight to one with the dropdown. Re-running the cell resets to the first case.

In [ ]:
import ipywidgets as widgets
from IPython.display import display, clear_output

case_labels = [
    f"{i:02d}  {r['gradcam_target']:6s}  {r['name']}  (dice={r['gradcam_best_dice']:.4f})"
    for i, r in enumerate(gradcam_results)
]

dropdown = widgets.Dropdown(options=list(zip(case_labels, range(len(gradcam_results)))),
                             description="case:", layout=widgets.Layout(width="600px"))
prev_btn = widgets.Button(description="< Prev")
next_btn = widgets.Button(description="Next >")
out = widgets.Output()

def render(idx):
    with out:
        clear_output(wait=True)
        show_case(gradcam_results[idx])

def on_dropdown_change(change):
    if change["name"] == "value":
        render(change["new"])

def on_prev(_):
    dropdown.value = max(0, dropdown.value - 1)

def on_next(_):
    dropdown.value = min(len(gradcam_results) - 1, dropdown.value + 1)

dropdown.observe(on_dropdown_change, names="value")
prev_btn.on_click(on_prev)
next_btn.on_click(on_next)

display(widgets.HBox([prev_btn, next_btn, dropdown]), out)
render(dropdown.value)


## 7. Aggregate stats


In [ ]:
summary_df = pd.DataFrame([
    {k: v for k, v in r.items() if k not in ("cam", "crop_info", "meta_row", "case")}
    for r in gradcam_results
])
summary_df.to_csv(OUT_DIR / "gradcam_summary.csv", index=False)

print(f"n cases          : {len(summary_df)}")
print(f"avg best dice     : {summary_df['gradcam_best_dice'].mean():.4f}")
print(f"pointing accuracy : {summary_df['gradcam_pointing_hit'].mean():.2%}")
summary_df


## Appendix: calibration note

The classifier's raw sigmoid probabilities cluster tightly around 0.5 for every
pathology on every scan — the classifier head's weights are small in magnitude, so
logits stay close to 0. That is a **calibration** artifact, not a broken model: AUROC
computed against real CT-RATE labels is meaningfully above chance (see
`models/ct-clip/scripts/eval_predictions_official.py`, which reuses the original
`eval.py::choose_operating_point` Youden's-J logic rather than inventing a new
threshold). Rank by raw logit (as this notebook does for case selection), not by
`prob >= 0.5`, if you need a meaningful ordering.
